# 蜃景 × lightx2v —— 纯文生视频 (t2v) Colab 部署（无 ComfyUI）

**怎么用：菜单「代码执行程序 → 全部运行」(Run all)。** 断线/回收后再 Run all。
- **Wan 权重(67G)直接下到本地 `/content`**（HF CDN ~170MB/s，比 Drive FUSE 读快几倍；本地 SSD 加载秒级）。代价=每会话重下(~10 分钟)，但整体比挂 Drive 加载快。
- lightx2v 每会话重装（很快，`--no-deps`）；SageAttention 现编译（可选，几分钟）。

**跑前一次性：**
1. 运行时选 **GPU**（Blackwell / Hopper / Ada / A100 都行）。
2. 右侧 🔑 Secrets 可加 **HF_TOKEN**（下 Wan 权重快些；公开模型匿名也能下）。
3. 第 1 格 `DEEPSEEK_KEY` 可留空（分镜 / AI 分析用前端配的 grok 模型）。

**§3/§4 已把 2026-06-17 实战踩平的坑固化（不用再手动改）：**
- lightx2v 0.1.0 钉死 `torch<2.8.0` 撞 cu128 torch 2.8 → 本体&依赖全 `--no-deps` 装。
- torch↔torchvision ABI 不匹配 → 子进程探测、坏了原子重装匹配的 cu128 三件套。
- `worldmirror`/`ring_attn` 跟单卡 t2v 无关却崩 import → 自动 patch 文件。
- editable `.pth` 内核不重读 → 一切**以干净子进程验证**。
- 配置**必须用 `configs/wan22/wan_moe_t2v.json`**（含 `boundary=0.875` 双专家切换点）；`wan/wan_t2v.json` 是旧单模型会崩 `KeyError 'boundary'`。§3 自动选对。
- 注意力默认 `torch_sdpa`（稳）；配置里默认的 `flash_attn3` 没装、调到会 `'NoneType' not callable` 崩，§3 已改掉；SageAttention 编上了用 `sage_attn2`（快）。
- **★`rope_type` 默认 `flashinfer`（没装）也会 `'NoneType' not callable`** → §3 已强制 `rope_type=torch`（出片跑通的最后一把钥匙，RTX PRO 6000 实测）。

**runner** = `wan2.2_moe`（A14B 双专家），`cpu_offload` 按显存自动（RTX PRO 6000 96G 实测全 GPU bf16；5090 32G 自动卸载）。**前端纯 t2v**：粘小说 → 一键分析 → 每镜「出片(t2v)」。

In [ ]:
# === §1 参数 + Drive + GPU 探测 ===
import os, sys
DEEPSEEK_KEY = ''            # 分镜/AI 分析用;留空=用前端配的 grok
REPO_URL = 'https://github.com/aw3n1998/build-with-langchain.git'; BRANCH = 'main'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
except Exception:
    pass
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'; CACHE = DRIVE + '/mirage_models'; TOOLS = DRIVE + '/mirage_tools'
for p in (CACHE, TOOLS):
    os.makedirs(p, exist_ok=True)
os.environ['HF_HOME'] = TOOLS + '/hf_cache'; os.makedirs(os.environ['HF_HOME'], exist_ok=True)
!nvidia-smi -L
import torch
if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    cc = torch.cuda.get_device_capability(0)
    os.environ['LX_CPU_OFFLOAD'] = 'false' if gb > 70 else 'true'   # >70G 两专家全 GPU;否则挪 CPU
    print(f'GPU {torch.cuda.get_device_name(0)} sm{cc[0]}.{cc[1]} {gb:.0f}G -> cpu_offload={os.environ["LX_CPU_OFFLOAD"]}')
    if cc[0] >= 12:
        print('  Blackwell(sm120): §3 编译 SageAttention(官方含 12.0)、patch worldmirror。')
else:
    os.environ['LX_CPU_OFFLOAD'] = 'true'; print('没检测到 GPU! 更改运行时类型 -> 选 GPU')
os.environ['PYTHONPATH'] = os.pathsep.join(p for p in sys.path if p)   # 子进程继承内核 torch
print('环境就绪 | 模型:', CACHE, '| 工具:', TOOLS)

In [ ]:
# === §2 拉取/更新 mirage 仓库 ===
import os, sys
%cd /content
if not os.path.isdir('mirage/.git'):
    !git clone -b {BRANCH} {REPO_URL} mirage
else:
    !cd mirage && git fetch origin {BRANCH} -q && git checkout {BRANCH} -q && git pull -q
%cd /content/mirage
sys.path.insert(0, '/content/mirage/colab')
print('mirage 就绪 @', BRANCH)

In [ ]:
# === §3 装 lightx2v（torch 锁 cu128 + --no-deps 绕 torch<2.8.0 冲突 + patch worldmirror/ring_attn + 选对 MoE 配置）===
# 今日实战踩平的坑(详见 git log / 笔记 lightx2v-t2v-blackwell-install):
#   ① lightx2v 0.1.0 钉 torch<2.8.0 撞 cu128-torch2.8 → 本体&依赖全 --no-deps 装;② 残留元数据→装前 uninstall;
#   ③ editable .pth 内核不重读→sys.path 加源码;④ worldmirror/ring_attn 崩 import→patch 文件;⑤ torch↔tv ABI→原子重装;
#   ⑥ 配置:必须用 configs/wan22/wan_moe_t2v.json(含 boundary=0.875 双专家切换点),configs/wan/wan_t2v.json 是旧单模型会崩 KeyError 'boundary'。
# 心法:一切以「干净子进程」import 为准(内核会被装卸污染;§5 server 本就是子进程)。
import os, subprocess, sys, json, re, glob, importlib.util as u
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout
def _ok(mod): return 'Traceback' not in sh(f'python -c "import {mod}" 2>&1')   # 子进程 import 成功=无 Traceback
LX = '/content/LightX2V'
# 1) clone + 源码加进 sys.path(editable 的 .pth 当前内核不重读)
if not os.path.isdir(LX + '/.git'):
    sh(f'git clone https://github.com/ModelTC/LightX2V.git {LX}')
if LX not in sys.path: sys.path.insert(0, LX)
# 2) torch 三件套:缺了或 torchvision ABI 坏了(子进程探测)→ 一条命令原子重装匹配的 cu128 套
if not _ok('torchvision'):
    print('torch/torchvision 缺或 ABI 不匹配 → 原子重装匹配的 cu128 三件套...')
    print(sh('pip install -q --force-reinstall --no-deps torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -3'))
v = sh("python -c 'import torch,torchvision,torchaudio;print(torch.__version__);print(torchvision.__version__);print(torchaudio.__version__)'").split()
con = '/content/torch_con.txt'; open(con, 'w').write('\n'.join(f'{p}=={x}' for p, x in zip(('torch', 'torchvision', 'torchaudio'), v[:3])))
print('torch 锁定:', v)
# 3) patch 跟单卡 t2v 无关、却崩 import 的两处(改文件→server 子进程也生效;幂等)
fa = LX + '/lightx2v/models/networks/worldmirror/models/layers/attention.py'
if os.path.exists(fa):
    s = open(fa).read()
    s = s.replace('from flash_attn_interface import flash_attn_func as flash_attn_func_v3', 'flash_attn_func_v3 = None  # [patch]')
    s = s.replace('from flash_attn.flash_attn_interface import flash_attn_func as flash_attn_func_v2', 'flash_attn_func_v2 = None  # [patch]')
    open(fa, 'w').write(s)
fr = LX + '/lightx2v/common/ops/attn/ring_attn.py'
if os.path.exists(fr):
    s = open(fr).read()
    s = re.sub(r'@torch\.jit\.script', '# @torch.jit.script  # [patch]', s)
    s = re.sub(r'torch\.jit\.script\(\s*([A-Za-z_]\w*)\s*\)', r'\1', s)
    open(fr, 'w').write(s)
# 4) 装 lightx2v(子进程 import 不过才装):本体 --no-deps 绕 torch<2.8.0 冲突;依赖也 --no-deps 装
if not _ok('lightx2v'):
    sh('pip uninstall -y lightx2v')
    print('装 lightx2v 本体(--no-deps,绕 torch<2.8.0 冲突)...'); print(sh(f'cd {LX} && pip install -e . --no-deps 2>&1 | tail -3'))
    import importlib.metadata as md
    try: reqs = md.requires('lightx2v') or []
    except Exception: reqs = []
    deps = [r.split(';')[0].strip() for r in reqs if 'extra ==' not in r and 'extra==' not in r]
    deps = [d for d in deps if d and not d.lower().startswith('torch')]
    open('/content/lx_deps.txt', 'w').write('\n'.join(deps))
    print(f'补装依赖({len(deps)} 个,--no-deps -c con)...'); print(sh(f'pip install -r /content/lx_deps.txt --no-deps -c {con} 2>&1 | tail -3'))
    NAME = {'cv2': 'opencv-python', 'PIL': 'pillow', 'yaml': 'pyyaml', 'sklearn': 'scikit-learn', 'skimage': 'scikit-image'}
    for _ in range(15):
        err = sh('python -c "import lightx2v" 2>&1')
        if 'Traceback' not in err: break
        m = re.search(r"No module named '([\w.]+)'", err)
        if not m: print('⚠ import 非缺包错:', err[-300:]); break
        mod = m.group(1).split('.')[0]; pkg = NAME.get(mod, mod); print('缺', mod, '→装', pkg); sh(f'pip install {pkg} --no-deps -c {con}')
# 5) SageAttention(可选提速;失败自动退 torch_sdpa)
if not _ok('sageattention'):
    if not os.path.isdir('/content/SageAttention/.git'):
        sh('git clone https://github.com/thu-ml/SageAttention.git /content/SageAttention')
    print('编译 SageAttention(sm120,约 3-8 分钟;失败退 torch_sdpa)...')
    print(sh(f'cd /content/SageAttention && CUDA_ARCHITECTURES="8.0,8.6,8.9,9.0,12.0" EXT_PARALLEL=4 '
             f'NVCC_APPEND_FLAGS="--threads 8" MAX_JOBS=32 pip install -e . --no-deps -c {con} 2>&1 | tail -5'))
_sage = _ok('sageattention')
# 6) 选对 MoE 配置(含 boundary)+ 套注意力/显存策略,写出 /content 供 §5 用(★兼容 5090:小卡 cpu_offload+model级)
_cands = sorted(set(glob.glob(f'{LX}/configs/**/wan_moe_t2v*.json', recursive=True)))
src_cfg = next((c for c in _cands if 'boundary' in json.load(open(c)) and 'lora' not in c.lower() and 'distill' not in c.lower()), None) \
       or next((c for c in _cands if 'boundary' in json.load(open(c))), None) or f'{LX}/configs/wan/wan_t2v.json'
c = json.load(open(src_cfg))
_attn = 'sage_attn2' if _sage else 'torch_sdpa'
for k in ('self_attn_1_type', 'cross_attn_1_type', 'cross_attn_2_type'): c[k] = _attn
_off = (os.environ.get('LX_CPU_OFFLOAD', 'true') == 'true')   # §1 按显存:>70G=False(双专家全GPU)/≤70G=True(卸载)
c['cpu_offload'] = _off
if _off:   # 小卡(5090 32G):cpu_offload + 模型级粒度;仍 OOM 把 offload_granularity 改 'block'→'phase'
    c['offload_granularity'] = 'model'; c['offload_ratio'] = 1.0
    c['t5_cpu_offload'] = False; c['vae_cpu_offload'] = False
else:
    c['offload_granularity'] = 'block'
# fp8(可选默认关):lightx2v 无现成 T2V-A14B fp8 权重,要 tools/convert/converter.py --linear_type fp8 自转双专家+sgl-kernel,
#   且 merge LoRA 与量化互斥(挂人物 LoRA 必须 bf16)。开法:c['dit_quantized']=True; c['dit_quant_scheme']='fp8-sgl'; c['dit_quantized_ckpt']='<fp8目录>'
c['rope_type'] = 'torch'        # ★出片 NoneType 真凶:默认 flashinfer 没装(--no-deps)→apply_rope=None→调它崩;强制 torch
c['rms_norm_type'] = 'torch'    # 防御:默认可能选未装的 sgl-kernel,强制 torch 稳
# ★出片时长:把帧长写进 config(治 config-only 那种 server「调帧数还出5秒」);只覆盖 config 已有的帧长键(不新增未知键,免 server 校验失败)。
_NF = int(os.environ.get('LIGHTX2V_NUM_FRAMES', '81') or 81)   # 想默认更长改这个(81≈5s/121≈7.5s/161≈10s,须 4n+1)
_nfk = [k for k in ('target_video_length', 'num_frames', 'video_length', 'num_frame', 'target_frames') if k in c]
for _k in _nfk: c[_k] = _NF
print('帧长键:', _nfk or '(config 无帧长键 → 帧数走 per-request)', '→', _NF, '帧')
USE_CFG = '/content/wan_moe_t2v_use.json'; json.dump(c, open(USE_CFG, 'w'), indent=2)
os.environ['LX_CONFIG'] = USE_CFG
# 7) 终检(干净子进程为准=§5 server 启动环境)
print('—' * 10)
print('lightx2v import:', '✅ OK' if _ok('lightx2v') else '❌ 看上面报错', '| 配置:', src_cfg.replace(LX + '/', ''),
      '| boundary:', c.get('boundary'), '| SageAttention:', _sage, '| attn:', _attn, '| cpu_offload:', _off)


In [ ]:
# === §4 下权重到本地 /content（直接从 HF 下到本地 SSD：比 Drive FUSE 读快几倍 + 本地加载秒级）===
# 取舍(用户定):本地盘每会话重下 ~67G(HF CDN 快,~10 分钟),换加载快、无 Drive FUSE 卡顿。
#   想持久省流量→把 MODEL 指回 Drive 的 CACHE 路径(但加载会慢)。LoRA 小,仍放 Drive 持久。
import os, glob
from huggingface_hub import snapshot_download
MODEL = '/content/wan_local'; LORA = CACHE + '/lightx2v_t2v_lora'
os.makedirs(MODEL, exist_ok=True); os.makedirs(LORA, exist_ok=True)
_have = (len(glob.glob(MODEL + '/high_noise_model/*.safetensors')) == 6 and len(glob.glob(MODEL + '/low_noise_model/*.safetensors')) == 6)
if not _have:
    print('从 HF 下 Wan2.2-T2V-A14B 到本地(~67G,并行8线程,断点续传)...')
    snapshot_download('Wan-AI/Wan2.2-T2V-A14B', local_dir=MODEL, max_workers=8,
        allow_patterns=['high_noise_model/*', 'low_noise_model/*', '*.json', 'Wan2.1_VAE.pth', 'models_t5_umt5-xxl-enc-bf16.pth', 'google/*'])
else:
    print('本地已有完整权重,跳过下载。')
# 4步蒸馏 LoRA(小,提速必挂)→Drive 持久
_LORA = 'Wan2.2-T2V-A14B-4steps-lora-rank64-Seko-V2.0'
snapshot_download('lightx2v/Wan2.2-Lightning', local_dir=LORA, allow_patterns=[f'{_LORA}/*'])
# 蒸馏 LoRA 双专家 high/low 路径 → 环境变量(供 §5d 挂载 + §6 .env);文件名 glob 实际为准(别写死)
def _find1(d, kw):
    fs = sorted(glob.glob(f'{d}/*{kw}*.safetensors')) or sorted(glob.glob(f'{d}/**/*{kw}*.safetensors', recursive=True))
    return fs[0] if fs else ''
_dld = f'{LORA}/{_LORA}'
os.environ['LIGHTX2V_DISTILL_LORA_HIGH'] = _find1(_dld, 'high')
os.environ['LIGHTX2V_DISTILL_LORA_LOW'] = _find1(_dld, 'low')
hi = len(glob.glob(f'{MODEL}/high_noise_model/*.safetensors')); lo = len(glob.glob(f'{MODEL}/low_noise_model/*.safetensors'))
os.environ['LIGHTX2V_MODEL_T2V'] = MODEL
print('本地权重 high/low:', hi, '/', lo, '✅' if (hi == 6 and lo == 6) else '❌ 缺→重跑本格(续传)', '| MODEL =', MODEL)
print('蒸馏 LoRA high/low:', os.environ['LIGHTX2V_DISTILL_LORA_HIGH'] or '(未找到→检查仓库结构)', '/', os.environ['LIGHTX2V_DISTILL_LORA_LOW'] or '(未找到)')

In [ ]:
# === §5 起 lightx2v server（model_cls=wan2.2_moe；端口 8189；脱离内核,停 cell 不杀）===
import os, subprocess, sys, glob, json
sys.path.insert(0, '/content/mirage/colab'); import persist
LX = '/content/LightX2V'
# 权重源:优先本地(§4 下到本地=秒载),否则退 Drive
_local = '/content/wan_local'
MODEL = _local if (len(glob.glob(_local + '/high_noise_model/*.safetensors')) == 6 and len(glob.glob(_local + '/low_noise_model/*.safetensors')) == 6) else CACHE + '/wan2.2_t2v_a14b'
_isloc = (MODEL == _local)
# 配置:用 §3 选好的 MoE 配置(含 boundary;§3 写到 LX_CONFIG);兜底自己找一个含 boundary 的
CFG = os.environ.get('LX_CONFIG', '')
if not (CFG and os.path.exists(CFG)):
    CFG = next((c for c in sorted(glob.glob(f'{LX}/configs/**/wan_moe_t2v*.json', recursive=True)) if 'boundary' in json.load(open(c))), f'{LX}/configs/wan22/wan_moe_t2v.json')
# ★把出片真帧挖出来:lightx2v server base.py 的 `raise exc` 是重抛(真错那帧被藏)→在重抛前 dump 完整 traceback。
#   幂等;专治「出片报 'NoneType' object is not callable 但看不到哪行 None」。重起后出片即可 grep REAL_TB /content/lightx2v.log。
for _bp in glob.glob(LX + '/lightx2v/server/**/base.py', recursive=True):
    _s = open(_bp).read()
    if 'REAL_TB' not in _s and 'raise exc' in _s:
        open(_bp, 'w').write(_s.replace('raise exc',
            'import traceback as _t; print("REAL_TB\\n"+"".join(_t.format_exception(type(exc), exc, exc.__traceback__)), flush=True); raise exc', 1))
        print('[patch] 出片真 traceback 已插桩:', _bp.replace(LX + '/', ''))
subprocess.run("fuser -k 8189/tcp 2>/dev/null; pkill -9 -f 'lightx2v.server' 2>/dev/null; true", shell=True)  # 硬重启
e = dict(os.environ); e['CUDA_VISIBLE_DEVICES'] = '0'; e['PYTHONUNBUFFERED'] = '1'
open('/content/lightx2v.log', 'w').close()
subprocess.Popen(['python', '-u', '-m', 'lightx2v.server', '--model_cls', 'wan2.2_moe', '--task', 't2v',
                  '--model_path', MODEL, '--config_json', CFG,
                  '--host', '0.0.0.0', '--port', '8189'],
                 cwd=LX, env=e, stdout=open('/content/lightx2v.log', 'w'), stderr=subprocess.STDOUT, start_new_session=True)
print('lightx2v server 起中 | 权重:', '本地 SSD(秒载) ✅' if _isloc else 'Drive(慢)', '| 配置:', CFG.replace(LX + '/', ''))
ok = persist.wait_http('http://127.0.0.1:8189/v1/service/status', timeout=(240 if _isloc else 900))
print('✅ lightx2v 就绪' if ok else '✗ 还没就绪 → 别重跑本格(会重载)!跑 §5c 查进度')
print(subprocess.run('tail -40 /content/lightx2v.log', shell=True, capture_output=True, text=True).stdout)
print('\n★崩 KeyError boundary → 配置用错了(该用 configs/wan22/wan_moe_t2v.json,不是 wan/wan_t2v.json);崩 patch_embedding → 跑 §5b。')
print('★裸片(无 LoRA)先验证通;出片报 NoneType → grep REAL_TB /content/lightx2v.log 看真帧。裸片通了再跑 §5d 挂 LoRA。')

In [ ]:
# === §5b (仅当 §5 崩在「KeyError patch_embedding / 加载权重」时跑) diffusers→native 转格式 ===
import glob
LX = '/content/LightX2V'; MODEL = '/content/wan_local'   # §4 下到本地的权重
for d in ('high_noise_model', 'low_noise_model'):
    !cd {LX} && python tools/convert/converter.py --source {MODEL}/{d}/ --output {MODEL}/{d}/ --output_ext .safetensors --output_name wan_{d} --model_type wan_dit --single_file
f = glob.glob(f'{MODEL}/high_noise_model/wan_*.safetensors')
if f:
    from safetensors import safe_open
    print('转后 key 样例:', list(safe_open(f[0], 'pt').keys())[:5])
print('转完 → 回 §5 重起 server')

In [ ]:
# === §5c 查 server 状态（不重启！server 加载 67G 慢,用这个看进度,别重跑 §5）===
import subprocess
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout
alive = bool(sh("pgrep -f 'lightx2v.server'").strip())
st = sh("curl -s -m 5 http://127.0.0.1:8189/v1/service/status") or '(还没响应,可能仍在加载)'
print('① server 进程:', '活着 ✅' if alive else '没了 ❌(崩了→看下面日志尾有无 Traceback)')
print('② status 端点:', st)
print('③ 日志尾 30 行:')
print(sh('tail -30 /content/lightx2v.log'))
print('\n判断: 进程活+status有响应=就绪(去 §6/§8/§9);进程活+无响应+日志在动无Traceback=还在加载(等几分钟再跑本格);进程没了+有Traceback=崩了。')

In [ ]:
# === §5d（可选；裸片通了再跑）挂角色/蒸馏 LoRA 进 server config → 重起 server ===
# 心法:lightx2v 的 LoRA 必须写进「起 server 的 config」才生效——per-request 传会被 server 忽略,改 LoRA 必须重起 server。
#   name 路由键只能 "high_noise_model"/"low_noise_model"(双专家各一条);同 name 多条会叠加(蒸馏加速 + 人物各一对)。
#   ★merge LoRA 与量化/lazy_load 互斥 → 挂 LoRA 必须 bf16(别同时开 fp8)。
# 角色 LoRA:在前端「角色 & LoRA」用 colab_deploy.ipynb 的 LW1/LW2 训出 char_lora_high/low_noise.safetensors 后,
#   把路径设进环境(下面两行),再跑本格;没训角色 LoRA 也能只挂蒸馏 LoRA(纯提速、人物不稳)。
import os, json, subprocess, glob, sys
sys.path.insert(0, '/content/mirage/colab'); import persist
LX = '/content/LightX2V'
# 角色 LoRA(训完填路径;留空=只挂蒸馏):
os.environ.setdefault('WAN_T2V_LORA_HIGH', '')   # 例:/content/drive/MyDrive/mirage_models/.../char_lora_high_noise.safetensors
os.environ.setdefault('WAN_T2V_LORA_LOW',  '')
CFG = os.environ.get('LX_CONFIG') or '/content/wan_moe_t2v_use.json'
c = json.load(open(CFG))
# ★角色 LoRA 强度可调(默认 1.0):出片【身份太淡/像别人】→ 先确认 caption 只写罕见触发词(后端已自动);仍淡→把
#   质量档 infer_steps 提到 6-8(给身份更多步数,比硬拉强度更稳),或这里上调 chi/clo 到 1.1~1.2;画面变形→下调到 0.7~0.9。
chi = float(os.environ.get('WAN_T2V_LORA_STR_HIGH', '1.0') or 1.0)   # 角色 high 噪专家强度(结构/身份)
clo = float(os.environ.get('WAN_T2V_LORA_STR_LOW',  '1.0') or 1.0)   # 角色 low 噪专家强度(细节/脸)
triples = [('high_noise_model', os.environ.get('WAN_T2V_LORA_HIGH', '').strip(), chi),       # 角色 high
           ('low_noise_model',  os.environ.get('WAN_T2V_LORA_LOW', '').strip(),  clo),       # 角色 low
           ('high_noise_model', os.environ.get('LIGHTX2V_DISTILL_LORA_HIGH', '').strip(), 1.0),# 蒸馏 high(§4,固定1.0)
           ('low_noise_model',  os.environ.get('LIGHTX2V_DISTILL_LORA_LOW', '').strip(), 1.0)] # 蒸馏 low(§4,固定1.0)
loras = [{'name': n, 'path': p, 'strength': s} for n, p, s in triples if p and os.path.exists(p)]
if not loras:
    print('⚠ 没找到任何 LoRA 文件。蒸馏 LoRA 应由 §4 下好;角色 LoRA 训完把路径填进上面两行 setdefault。')
else:
    c['lora_configs'] = loras
    c['dit_quantized'] = False   # 挂 LoRA 必须 bf16(与量化互斥)
    c['infer_steps'] = int(os.environ.get('LIGHTX2V_STEPS', '4') or 4)   # 画质档:极速4/成片8(改这个再跑本格重起生效)
    _NF = int(os.environ.get('LIGHTX2V_NUM_FRAMES', '81') or 81)         # 出片时长(81≈5s/121≈7.5s/161≈10s,须 4n+1)
    for _k in ('target_video_length', 'num_frames', 'video_length', 'num_frame', 'target_frames'):
        if _k in c: c[_k] = _NF
    c['enable_cfg'] = False               # 蒸馏不用 CFG,关了每步再省一半
    c['sample_guide_scale'] = [1.0, 1.0]  # ★MoE 双专家各取一个 [0]/[1],必须是 2 元素列表;标量会崩 'float' object is not subscriptable
    CFG_LORA = '/content/wan_moe_t2v_lora.json'; json.dump(c, open(CFG_LORA, 'w'), indent=2)
    os.environ['LX_CONFIG'] = CFG_LORA   # 让之后重跑 §5 也用带 LoRA 的 config
    print('lora_configs 写入 server config:\n' + json.dumps(loras, ensure_ascii=False, indent=2))
    MODEL = os.environ.get('LIGHTX2V_MODEL_T2V', '/content/wan_local')
    subprocess.run("fuser -k 8189/tcp 2>/dev/null; pkill -9 -f 'lightx2v.server' 2>/dev/null; true", shell=True)
    e = dict(os.environ); e['CUDA_VISIBLE_DEVICES'] = '0'; e['PYTHONUNBUFFERED'] = '1'
    open('/content/lightx2v.log', 'w').close()
    subprocess.Popen(['python', '-u', '-m', 'lightx2v.server', '--model_cls', 'wan2.2_moe', '--task', 't2v',
                      '--model_path', MODEL, '--config_json', CFG_LORA, '--host', '0.0.0.0', '--port', '8189'],
                     cwd=LX, env=e, stdout=open('/content/lightx2v.log', 'w'), stderr=subprocess.STDOUT, start_new_session=True)
    ok = persist.wait_http('http://127.0.0.1:8189/v1/service/status', timeout=300)
    print('✅ 带 LoRA 重起就绪' if ok else '✗ 未就绪 → 跑 §5c 查;崩则看是否量化没关 / LoRA key 不匹配(grep "lora key not loaded" 日志)')

In [ ]:
# === §6 装后端依赖 + 写 .env（纯 t2v：COMFYUI 空、lightx2v 接通）===
import os, re, shutil
%cd /content/mirage
!pip -q install -r requirements.txt
!pip -q install fastembed edge-tts
shutil.copy('.env.colab', '.env'); env = open('.env').read()
def _se(e, k, v): return re.sub(k + r'=.*', k + '=' + v, e) if (k + '=') in e else e + ('' if e.endswith(chr(10)) else chr(10)) + k + '=' + v + chr(10)
MODEL = '/content/wan_local'   # §4/§5 用的本地权重
# 蒸馏 4 步 LoRA(§4 已 glob 出实际文件名) → 登记到 LIGHTX2V_DISTILL_LORA_*(不是角色槽!);角色 LoRA 走 WAN_T2V_LORA_*(训完再填)
_dhi = os.environ.get('LIGHTX2V_DISTILL_LORA_HIGH', '')
_dlo = os.environ.get('LIGHTX2V_DISTILL_LORA_LOW', '')
for k, v in [('COMFYUI_BASE_URL', ''), ('LIGHTX2V_ENABLED', 'true'), ('LIGHTX2V_BASE_URL', 'http://127.0.0.1:8189'),
             ('LIGHTX2V_MODEL_T2V', MODEL), ('T2V_PROVIDER', 'lightx2v-t2v'), ('VIDEO_PROVIDER_DEFAULT', 'wan2.2'),
             ('LIGHTX2V_DISTILL_LORA_HIGH', _dhi), ('LIGHTX2V_DISTILL_LORA_LOW', _dlo),
             ('COMFYUI_LORA_DIR', CACHE + '/char_loras'),                              # ★训好的角色 LoRA 自动存 Drive(持久,免 Colab 回收丢)
             ('LORA_TRAIN_BASE', 'ai-toolkit/Wan2.2-T2V-A14B-Diffusers-bf16'),         # Wan2.2-T2V 训练底模(不是 FLUX;前端训练用)
             ('AI_TOOLKIT_DIR', '/content/ai-toolkit')]:                                # ai-toolkit 位置(前端「开始训练」需先装它,见可选安装格)
    env = _se(env, k, v)
if DEEPSEEK_KEY:
    env = _se(env, 'OPENAI_API_KEY', DEEPSEEK_KEY)
open('.env', 'w').write(env)
print('.env 写好(纯 t2v: COMFYUI 空 / lightx2v 接通 / 蒸馏 LoRA 登记 LIGHTX2V_DISTILL_LORA_*)')
print('LoRA 挂载:权威路径是 §5d 把 lora_configs 写进 server config(per-request 传会被 server 忽略)。')
print('  后端 provider 接入已修(条目带 name=high/low_noise_model、前向兼容);角色 LoRA 训完填 WAN_T2V_LORA_HIGH/LOW 再跑 §5d。')

In [ ]:
# === §7 构建前端（纯 t2v 工作台）===
%cd /content/mirage/frontend
!npm install --silent && npm run build || echo '前端构建失败(用 API 也行)'
%cd /content/mirage

In [ ]:
# === §8 起后端（硬重启：杀旧+起新，读 §6 的 .env）===
import sys; sys.path.insert(0, '/content/mirage/colab'); import persist
persist.restart_service('后端', ['uvicorn', 'mirage.main_api:app', '--host', '0.0.0.0', '--port', '8000'],
    'http://127.0.0.1:8000/api/health', '/content/api.log', 8000, 'uvicorn', cwd='/content/mirage', timeout=120)

In [ ]:
# === §9 cloudflared 暴露后端 → 公网 URL ===
import subprocess, re, time, os, sys
sys.path.insert(0, '/content/mirage/colab'); import persist
subprocess.run('pkill -9 -f cloudflared 2>/dev/null; sleep 2', shell=True)
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True)
open('/content/cf.log', 'w').close()
persist.start_bg(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], '/content/cf.log')
url = None
for _ in range(60):
    time.sleep(1)
    m = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cf.log').read())
    if m:
        url = m[-1]; break
print('✅ 公网地址:', url if url else '⚠ 60s 未就绪,重跑本格')
print('打开它 → 工作台纯 t2v: 粘小说 → 一键分析 → 每镜「出片(t2v)」')

In [ ]:
# === §10 实时日志（lightx2v 出片采样进度；停本格只停 tail、不杀 server）===
!tail -n 40 -f /content/lightx2v.log

In [ ]:
# === §日志速查（一键体检：跑一下看各服务+日志现状；下面注释是按需单独跑的命令）===
import subprocess
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout
print('【进程】 lightx2v:', '活✅' if sh("pgrep -f lightx2v.server").strip() else '没了❌',
      '| 后端:', '活✅' if sh("pgrep -f mirage.main_api").strip() else '没了❌',
      '| cloudflared:', '活✅' if sh("pgrep -f cloudflared").strip() else '没了❌')
print('【lightx2v 状态】', (sh("curl -s -m5 http://127.0.0.1:8189/v1/service/status") or '(无响应/加载中)').strip()[:200])
print('【公网 URL】', (sh(r"grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /content/cf.log | tail -1") or '(还没有)').strip())
print('【出片真帧 REAL_TB（NoneType 调试）】')
print(sh("grep -A40 REAL_TB /content/lightx2v.log | tail -45") or '  (没有 REAL_TB → 还没崩过/还没出片)')
print('【lightx2v.log 尾 25 行】'); print(sh("tail -25 /content/lightx2v.log"))
print('【后端 api.log 尾 12 行】'); print(sh("tail -12 /content/api.log"))
# ── 按需单独跑(复制到新格)──────────────────────────
# 实时盯出片采样(会一直刷,停本格即停): !tail -n 80 -f /content/lightx2v.log
# 只挖出片真帧(NoneType 定位)        : !grep -A60 REAL_TB /content/lightx2v.log
# 挂 LoRA 后查 key 不匹配            : !grep -iE "lora.*(not loaded|missing|mismatch|skip)" /content/lightx2v.log
# 看当前生效配置(attn/boundary/offload): import json; print(json.load(open('/content/wan_moe_t2v_use.json')))
# GPU 占用 / 后端实时日志            : !nvidia-smi   /   !tail -n 60 -f /content/api.log
# 验证帧数是否 per-request 生效   : 出两条不同「时长」的片,看 api.log 的「num_frames=」与实际出片秒数是否一致;若改时长出片仍 5s=server 按 config 锁帧长,改 §3/§5d 的 LIGHTX2V_NUM_FRAMES 重起


---
## 🎉 经验总结（RTX PRO 6000 实测跑通）

**一句话**：lightx2v 装机坑 §3 全固化了；出片成败的关键是**别让任何算子默认到没装的后端**——`attn` 三键 + **`rope_type`** 都得强制 `torch`，否则报 `'NoneType' object is not callable`。

**验证硬件**：RTX PRO 6000（Blackwell sm120, 96G）→ `cpu_offload=False`、双专家全 GPU bf16；SageAttention 在 sm120 编不过（已知：用 Hopper wgmma 指令）→ 自动退 `torch_sdpa`，不影响出片。

### 安装坑（§3 已自动处理，不用手动改）
1. lightx2v 0.1.0 钉 `torch<2.8.0`，撞 Colab cu128 torch2.8 → 本体&依赖全 `--no-deps` 装；**别 `pip install lightx2v`**（PyPI 是残壳，装前先 uninstall + editable）。
2. editable `.pth` 当前内核不重读 → 一切以**干净子进程** import 为准。
3. `worldmirror`（硬 import flash_attn）/ `ring_attn`（`@torch.jit.script` 新 torch 编不过）跟单卡 t2v 无关却崩 import → patch 文件。
4. torch↔torchvision ABI 不匹配 → 子进程探测、坏了原子重装匹配的 cu128 三件套。

### 配置坑（出片成败的关键，§3 已固化）
5. **必须用 `configs/wan22/wan_moe_t2v.json`**（含 `boundary=0.875` 双专家切换点）；`wan/wan_t2v.json` 是旧单模型 → 崩 `KeyError 'boundary'`。
6. **`attn` 三键**（`self_attn_1_type`/`cross_attn_1_type`/`cross_attn_2_type`）→ `torch_sdpa`（`flash_attn3` 没装）。
7. **★`rope_type`** 默认 `flashinfer`（没装）→ `apply_rope...` 是 `None`，第一个 transformer block 调它就崩。**必须 `rope_type=torch`**——这是「attn 都改了仍崩」的最后一把钥匙；`rms_norm_type=torch` 防御性同钉。
8. 量化默认关（bf16）；挂 LoRA 与量化互斥，别同时开 fp8。
9. **★4 步极速档（出片提速 ~10×）**：挂蒸馏 LoRA + `infer_steps=4` + `enable_cfg=False` + **`sample_guide_scale` 必须是 `[v, v]` 两元素列表**（MoE 双专家各取一个 `[0]`/`[1]`；设成标量 `1.0` 会崩 `'float' object is not subscriptable` @ wan_runner.py get_current_model_index）。§5d 已固化；RTX PRO 6000 实测跑通。
10. **取片**：lightx2v 把片存 `<install>/lightx2v/server_cache/outputs/{task_id}.mp4`（同机本地）；mirage 已按此约定拷回（commit 02dda60），前端正常收片。

### 权重 / 加载
9. `snapshot_download` 直接下本地 `/content/wan_local`（比 Drive FUSE 快几倍，本地 SSD 加载秒级）；代价=每会话重下 ~67G。
10. diffusers 格式直读；只有崩 `patch_embedding` 才跑 §5b 转 native。

### 出片报错怎么挖真帧
11. 出片 `'NoneType' object is not callable`：`base.py:196` 是**重抛**、真帧跨进程丢；**真帧在 worker** → `grep -n 'inference failed' -A 120 /content/lightx2v.log`（`worker.py:108 logger.exception` 打的，在 `REAL_TB` **之前**）。看最深帧 `File ... line ...` 定位是哪个没装的算子，按同法把对应 `*_type` 强制 `torch`。

---
## 角色 LoRA 训练 & 挂载（Wan-T2V，可选）
- **训练**用 `colab_deploy.ipynb` 的 **LW1/LW2**（ai-toolkit，与 lightx2v 推理两套，互不影响）。产物 `char_lora_high_noise.safetensors` / `char_lora_low_noise.safetensors`。
- **挂载**：把产物路径填进 **§5d** 顶部的 `WAN_T2V_LORA_HIGH/LOW`，跑 §5d → 把 `lora_configs`（`name`=`high_noise_model`/`low_noise_model`，蒸馏 + 人物各一对、同 name 叠加）写进 server config 并重起 server。
  - LoRA **只在「起 server 的 config」里生效**（per-request 传会被忽略）；**改 LoRA 必须重起 server**；**merge LoRA 与量化互斥（必须 bf16，§5d 已自动关量化）**。


### ★出片「跟训练的人完全对不上 / 是别人」——头号坑：触发词
角色 LoRA 身份能否绑上，命门是**触发词必须是无预训练语义的罕见 token**。用 `char`/`person`/人名 这类常见词当触发词，
身份会糊进该词原有语义 → 出片渲染的是预训练里的「某个人」而非你训的人（症状：完全是别人）。
**后端已自动修这两点（2026-06-18）**，前端无需再操心：
1. 触发词留空或填了常见词 → 自动换成 `zq` 开头的罕见 token（`effective_trigger`）；打 caption 与出片注入统一用它。
2. 打 caption **只写触发词、绝不写外貌**（脸写进 caption 会和身份绑定打架）。外貌只在出片画面提示词里出现。
> 旧 LoRA（触发词 `char` + caption 带外貌）已被污染、救不回 → **必须重训**：前端「角色 & LoRA」重新上传同一批图（选中该角色，触发词留空），重训后跑 §5d 挂上。

### 仍偏淡 / 像但会飘 —— 是 T2V-LoRA 的天花板，不是 bug
- 干净配方下 Wan2.2-T2V 角色 LoRA 给的是**强可辨识度**（一看就是这个人），但**运镜/转头/大角度时脸会朝通用脸漂**——这是 T2V 角色 LoRA 的固有局限。
- 想更稳：① 出片用**中等质量档（infer_steps 6-8）**而非 4 步极速档（给身份更多步数，比硬拉强度稳）；② 角色强度在 §5d 用 `WAN_T2V_LORA_STR_HIGH/LOW` 微调（淡→1.1~1.2，变形→0.7~0.9）；③ 数据集凑到 ~25-30 张、多角度多光照。
- 要**精确到像素的同一张脸**：T2V-LoRA 做不到，那是图像模型（FLUX+PuLID）/ i2v 参考图 / 换脸后处理的领域。

## 首跑核对清单
1. **§1** 打印 `GPU ... smX.Y ...G -> cpu_offload=...`？没 GPU → 运行时改选 GPU。
2. **§3** 末行 `lightx2v import: ✅ OK | 配置: configs/wan22/wan_moe_t2v.json | boundary: 0.875 | attn: torch_sdpa`（且 §3 已强制 `rope_type=torch`——出片不崩 NoneType 的关键）？`SageAttention: False` 不影响。
3. **§4** `本地权重 high/low: 6 / 6 ✅` + `蒸馏 LoRA high/low: ...`？否 → 重跑 §4（断点续传）。
4. **§5** `✅ lightx2v 就绪`？先验**裸片**（无 LoRA）出片通不通。
5. **§6 → §8 → §9** .env 接通 → 后端起 → 公网 URL → 出片。
6. 裸片通了、想锁人物 → 训角色 LoRA → 填进 **§5d** 跑一次（重起 server）。